In [1]:
import pandas as pd
import sys
import os
import time
import joblib
import numpy as np

from tslearn.metrics import cdist_dtw

sys.path.insert(0, '../../..') # Load Parent directory to import utils
from utils.constants import Configs
from utils.helper_functions import normalize_series # calcula a distância entre séries temporais usando DTW

/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/home/hugo-guimar-es/Documents/GitHub/python_zeromq/Outlier_Removal/.venv/lib/python3.13/site-packages/tslearn/bases/bases.py:16: UserWarning: h5py not installed, hdf5 features will not be supported.
Install h5py to use hdf5 features: http://docs.h5py.org/
  warn(h5py_msg)


### Load Data From cache

In [2]:
cache_path = Configs.SMART_RETAIL_FILTERED_DATA_BIMANUAL
if not os.path.exists(cache_path):
    raise FileNotFoundError(f"Processed data not found at {cache_path}. Please run the data processing notebook first.")
df = pd.read_csv(cache_path)

### Delete Subject 13

In [3]:
# df.drop(df[df['sub']== 13].index, inplace=True)
# df[df['sub']==13].head()

### Calculate DTW Matrix for all movements
This approach assumes that the quality of a bimanual movement is decided by the worst performed arm. I'll use the distances of the left and right arm and only store the greatest between the two of them.
$$DTW_{final} = \max(DTW_{esq}, DTW_{dir})$$

ou:

$$DTW_{fused}(A, B) = \max \left( \min_{\pi_{esq}} \sqrt{ \sum_{(i,j) \in \pi_{esq}} \sum_{k \in \{x_L, y_L, z_L\}} (A_{i,k} - B_{j,k})^2 }, \min_{\pi_{dir}} \sqrt{ \sum_{(u,v) \in \pi_{dir}} \sum_{m \in \{x_R, y_R, z_R\}} (A_{u,m} - B_{v,m})^2 } \right)$$

Instead of tuning the DTW globally, I'll test diferent window_percentages and save them on cache. When LOSO is being performed, inside each fold there will be a selection of the best DTW based on the different saved DTW matrices. I can perform a tuning without exponential increasing the computational time of the LOSO cross-validation.

In [ ]:
group_left = ['x_L', 'y_L', 'z_L']
group_right = ['x_R', 'y_R', 'z_R']

sample_size = None
precompute_times = {} 

for movement in ["Pick", "Park", "Place"]:
    print(f"\n--- Tuning Constrained Bimanual DTW for {movement} ---")
    
    normalized_left = normalize_series(df_input=df, movLabel_filter=movement, useScaler=False,
                                       group_array=group_left, sample_size=sample_size)
    normalized_right = normalize_series(df_input=df, movLabel_filter=movement, useScaler=False,
                                        group_array=group_right, sample_size=sample_size)
    if normalized_left.shape[0] != normalized_right.shape[0]:
        raise ValueError("Number of samples between Left and Right do not match!")

    
    seq_length = normalized_left.shape[1]
    precompute_times[movement] = {}
    
    best_score = -1
    best_radius = None
    best_time = 0.0

    print(f"  -> Computing DTW matrices for {movement} with sequence length={seq_length} and sample size={sample_size}")
    for sec in Configs.DTW_RADIUS_Seconds:
        radius = max(1, round(sec * Configs.FS))  # Convert seconds to samples, ensuring at least 1 sample
        t0 = time.perf_counter()
        
        dtw_left = cdist_dtw(normalized_left, global_constraint="sakoe_chiba",
                             sakoe_chiba_radius=radius, n_jobs=-1)
        dtw_right = cdist_dtw(normalized_right, global_constraint="sakoe_chiba",
                              sakoe_chiba_radius=radius, n_jobs=-1)
        dtw_fused = np.maximum(dtw_left, dtw_right)     # late fusion

        precompute_times[movement][sec] = time.perf_counter() - t0

        
        # Export matrix
        cache_name = str(Configs.late_fusion_dtw_matrix_path(movement, sec))
        joblib.dump(dtw_fused, cache_name)
        print(f"  -> w={sec:.2f} | radius={radius} | {precompute_times[movement][sec]:.2f}s")        


--- Tuning Constrained Bimanual DTW for Pick ---


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 2884.82it/s]

Shape of first series: (106, 3)


Shape of formatted series: (380, 461, 3)


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 3040.76it/s]

Shape of first series: (106, 3)


... (8 outputs left out)

Shape of formatted series: (380, 2126, 3)


Processing time series: 100%|██████████| 380/380 [00:00<00:00, 3054.24it/s]

Shape of first series: (121, 3)


Shape of formatted series: (380, 2126, 3)
  -> computed DTW matrices for Place with radius=6 (sec=0.10) in 987.43s
  -> w=0.10 | radius=6 | 987.43s
  -> computed DTW matrices for Place with radius=12 (sec=0.20) in 983.34s
  -> w=0.20 | radius=12 | 983.34s
  -> computed DTW matrices for Place with radius=18 (sec=0.30) in 983.91s
  -> w=0.30 | radius=18 | 983.91s
  -> computed DTW matrices for Place with radius=24 (sec=0.40) in 983.95s
  -> w=0.40 | radius=24 | 983.95s
  -> computed DTW matrices for Place with radius=30 (sec=0.50) in 991.30s
  -> w=0.50 | radius=30 | 991.30s


In [5]:
precompute_times
for time in precompute_times:
    print(f"{time}: {precompute_times[time]}")

Pick: {0.1: 197.266856121, 0.2: 193.87114285300004, 0.3: 194.71986748000006, 0.4: 192.12027276100002, 0.5: 207.06433449099984}
Park: {0.1: 50.30415157100015, 0.2: 50.289816543999905, 0.3: 50.528249409999944, 0.4: 50.05027222099989, 0.5: 49.71558842200011}
Place: {0.1: 987.434081504, 0.2: 983.3383449500002, 0.3: 983.9070323619994, 0.4: 983.948014132, 0.5: 991.3049215599995}
